In [2]:
import sys
sys.path.append("..")
from benchmarks.guacamol.assess_goal_directed_generation import assess_goal_directed_from_smiles
import pandas as pd


conditions = {
    "alzheimer": ["AChE", "MAOB"],
    "schizophrenia": ["D2R", "_5HT2A"],
    "parkinson": ["D2R", "D3R"],
}

condition_labels = {
    "alzheimer": "Alzheimers",
    "schizophrenia": "Schizophrenia",
    "parkinson": "Parkinsons",
}

smiles_dict = {}

# Load generated molecules SMILES
for disease, targets in conditions.items():
    # targets_file = "_".join(targets) + "_SUM.csv"
    disease_label = condition_labels[disease]
    targets_file = f"{disease_label}_SUM.csv"
    smiles_list = pd.read_csv(f"../generated_molecules/25-epoch/{targets_file}")["SMILES"].tolist()
    smiles_dict[disease] = smiles_list

compo_gpt_results = assess_goal_directed_from_smiles(
    smiles_dict, json_output_file=None, benchmark_version="multitarget"
)
results_df = pd.DataFrame(compo_gpt_results["results"])
results_df["Approach"] = "CoMPO-GPT"
results_df

Running benchmark 1/3: alzheimer
alzheimer: 74 instead of 100
[WARNING] An incorrect number of distinct molecules was generated: 74 instead of 100. Padding scores with 26 zeros...
  Score: 0.742253
  Execution time: 0:00:00
Running benchmark 2/3: schizophrenia
schizophrenia: 87 instead of 100
[WARNING] An incorrect number of distinct molecules was generated: 87 instead of 100. Padding scores with 13 zeros...
  Score: 0.799669
  Execution time: 0:00:00
Running benchmark 3/3: parkinson
parkinson: 83 instead of 100
[WARNING] An incorrect number of distinct molecules was generated: 83 instead of 100. Padding scores with 17 zeros...
  Score: 0.795565
  Execution time: 0:00:00


,benchmark_name,score,optimized_molecules,execution_time,number_scoring_function_calls,metadata,Approach
0,alzheimer,0.742253,"[(CCC(=O)c1cccc(OC(=O)C(C)C)c1C, 0.84187934418...",0,74,"{'top_1': 0.8418793441801726, 'top_10': 0.8209...",CoMPO-GPT
1,schizophrenia,0.799669,"[(COc1ccc(C(=O)CCCC(C)C)cc1OC, 0.8661357786154...",0,87,"{'top_1': 0.8661357786154263, 'top_10': 0.8480...",CoMPO-GPT
2,parkinson,0.795565,"[(COc1ccccc1N1CCN(CCCCc2ccc3c(c2)CCO3)CC1, 0.8...",0,83,"{'top_1': 0.8847098455590622, 'top_10': 0.8497...",CoMPO-GPT


In [3]:
from pathlib import Path
import numpy as np

# Ensure runtime warnings raise as errors to debug
import warnings
warnings.filterwarnings(
    action='error', message='',
    category=RuntimeWarning
)


METHODS = {"CoMPO-GPT": compo_gpt_results}

# Baselines
baselines = [
    "DeepLig",
    "POLYGON",
    "MTMol-GPT",
]

diseases = ["alzheimer", "schizophrenia", "parkinson"]

for baseline in baselines:
    print(baseline)
    # List of diseases to evaluate
    diseases = ["alzheimer", "schizophrenia", "parkinson"]
    
    # Load generated molecules for each disease
    smiles_dict = {}
    
    for disease in diseases:
        path = f"../generated_molecules/{baseline}/{disease}-mpo.csv"
        if Path(path).exists():
            df = pd.read_csv(path,header=None)
            smiles_dict[disease] = df[0].tolist()
        else:
            print(f"Warning: No file found for {baseline} - {disease}")

    # Run benchmarks and store results
    METHODS[baseline] = assess_goal_directed_from_smiles(
        smiles_dict=smiles_dict,
        json_output_file=None, 
        benchmark_version="multitarget"
    )

METHODS.keys()

DeepLig
Running benchmark 1/3: alzheimer
alzheimer: 48 instead of 100
[WARNING] An incorrect number of distinct molecules was generated: 48 instead of 100. Padding scores with 52 zeros...
  Score: 0.646336
  Execution time: 0:00:00
Running benchmark 2/3: schizophrenia
schizophrenia: 73 instead of 100
[WARNING] An incorrect number of distinct molecules was generated: 73 instead of 100. Padding scores with 27 zeros...
  Score: 0.762679
  Execution time: 0:00:00
Running benchmark 3/3: parkinson
parkinson: 35 instead of 100
[WARNING] An incorrect number of distinct molecules was generated: 35 instead of 100. Padding scores with 65 zeros...
  Score: 0.664676
  Execution time: 0:00:00
POLYGON
Running benchmark 1/3: alzheimer
alzheimer: 99 instead of 100
[WARNING] An incorrect number of distinct molecules was generated: 99 instead of 100. Padding scores with 1 zeros...
  Score: 0.627868
  Execution time: 0:00:00
Running benchmark 2/3: schizophrenia
schizophrenia: 99 instead of 100
[WARNING] A

dict_keys(['CoMPO-GPT', 'DeepLig', 'POLYGON', 'MTMol-GPT'])

In [24]:
from benchmarks.guacamol.assess_distribution_learning import assess_distribution_learning_from_smiles

# Distribution learning results storage - flat list of results per method
DISTRIBUTION_METHODS = {}

# For CoMPO-GPT: load from {disease_label}_SUM.csv files
compo_gpt_dist_all_results = []

for disease, targets in conditions.items():
    disease_label = condition_labels[disease]
    targets_file = f"{disease_label}_SUM.csv"
    smiles_list = pd.read_csv(f"../generated_molecules/25-epoch/{targets_file}")["SMILES"].tolist()
    
    # Load training file for comparison
    training_file = f"../data/training_files/all_active_compounds_{disease}.smi"
    
    results = assess_distribution_learning_from_smiles(
        smiles_list,
        chembl_training_file=training_file,
        json_output_file=None,
        number_samples=2000
    )
    
    # Each result in results["results"] has benchmark_name and score
    # Add disease information to each result
    for result in results["results"]:
        result_copy = result.copy()
        result_copy["disease"] = disease
        result_copy["metadata"] = {result_copy["benchmark_name"]: result_copy["score"]}
        compo_gpt_dist_all_results.append(result_copy)

DISTRIBUTION_METHODS["CoMPO-GPT"] = compo_gpt_dist_all_results

# For baselines: load from {disease}-mpo.csv files
for baseline in baselines:
    baseline_dist_all_results = []
    
    for disease in diseases:
        path = f"../generated_molecules/{baseline}/{disease}-mpo.csv"
        if Path(path).exists():
            df = pd.read_csv(path, header=None)
            smiles_list = df[0].tolist()
            
            # Load training file for comparison
            training_file = f"../data/training_files/all_active_compounds_{disease}.smi"
            
            results = assess_distribution_learning_from_smiles(
                smiles_list,
                chembl_training_file=training_file,
                json_output_file=None,
                number_samples=2000
            )
            
            # Add disease information to each result
            for result in results["results"]:
                result_copy = result.copy()
                result_copy["disease"] = disease
                result_copy["metadata"] = {result_copy["benchmark_name"]: result_copy["score"]}
                baseline_dist_all_results.append(result_copy)
    
    DISTRIBUTION_METHODS[baseline] = baseline_dist_all_results

print("Distribution learning benchmarks completed")

Distribution learning benchmarks completed


In [25]:
metric_order = [
    "Score",
    "Target Response",
    "Blood-Brain Barrier",
    "CNS MPO",
    "Synthetic Accessibility",
    "Validity",
    "Uniqueness",
    "Novelty",
    "KL divergence",
    "Frechet ChemNet Distance",
]

benchmark_order = [
    "alzheimer",
    "schizophrenia",
    "parkinson",
]

report_results = list()

def get_metadata_keys(metadata):
    keys = list(metadata.keys())

    target = [k for k in keys if "GeometricMeanScoringFunction" in k]
    target_scores = metadata[target[0]] if len(target) > 0 else None

    bbb = [k for k in keys if "BBBResponseScoringFunction" in k]
    bbb_scores = metadata[bbb[0]] if len(bbb) > 0 else None

    sa = [k for k in keys if "SyntheticAccessibilityScoringFunction" in k]
    sa_scores = metadata[sa[0]] if len(sa) > 0 else None

    cns = [k for k in keys if "CNS_MPO_ScoringFunction" in k]
    cns_scores = metadata[cns[0]] if len(cns) > 0 else None

    return {
        "Target Response": target_scores,
        "Blood-Brain Barrier": bbb_scores,
        "Synthetic Accessibility": sa_scores,
        "CNS MPO": cns_scores        
    }


for method, report in METHODS.items():
    results = report["results"]
    distribution_results = DISTRIBUTION_METHODS[method]
        
    for result in results:
        results_info = dict()

        results_info["Method"] = method
        results_info["Benchmark"] = result["benchmark_name"]
        results_info["Score"] = f"{result['score']:.4f}"

        metadata = result["metadata"]
        scores_dict = get_metadata_keys(metadata)

        for score_key in scores_dict:
            scores = scores_dict[score_key]
            if scores is None:
                continue

            mean_score = np.mean(scores)
            std_score = np.std(scores)

            results_info[score_key] = f"{mean_score:.3f} ± {std_score:.1f}"
        
        # Find distribution result where result["disease"] == result["benchmark_name"]
        for dist_result in distribution_results:
            if dist_result["disease"] != result["benchmark_name"]:
                continue

            for metric_name in dist_result["metadata"]:
                results_info[metric_name] = f"{dist_result['metadata'][metric_name]:.3f}"

        report_results.append(results_info.copy())

# Reorder the benchmarks
df = pd.DataFrame(report_results)   
df["Benchmark"] = pd.Categorical(df["Benchmark"], benchmark_order, ordered=True)
dfs = list()

for metric in metric_order:
    df_score = df.pivot_table(
        index=["Benchmark"],
        columns=["Method"],
        values=[metric],
        aggfunc=lambda x: x,
    )

    df_score.columns = df_score.columns.droplevel(0)
    df_score["Metric"] = metric

    dfs.append(df_score)

concat_dfs = pd.concat(dfs, axis=0).sort_values(by=["Benchmark"]).reset_index()

concat_dfs["Metric"] = pd.Categorical(
    concat_dfs["Metric"], categories=metric_order, ordered=True
)

viz_df = concat_dfs.sort_values(by=["Benchmark", "Metric"]).set_index(["Benchmark", "Metric"])
viz_df

Method                                    CoMPO-GPT      DeepLig    MTMol-GPT   
Benchmark     Metric                                                            
alzheimer     Score                          0.7423       0.6463       0.7293  \
              Target Response           0.480 ± 0.1  0.363 ± 0.1  0.482 ± 0.1   
              Blood-Brain Barrier       0.809 ± 0.1  0.880 ± 0.1  0.624 ± 0.2   
              CNS MPO                   0.974 ± 0.0  0.896 ± 0.1  0.925 ± 0.1   
              Synthetic Accessibility   0.908 ± 0.0  0.949 ± 0.0  0.829 ± 0.1   
              Validity                        0.782        0.977        0.884   
              Uniqueness                      0.696        0.147        0.815   
              Novelty                         1.000        1.000        0.869   
              KL divergence                   0.194        0.229        0.849   
              Frechet ChemNet Distance        0.001        0.001        0.170   
schizophrenia Score                          0.7997       0.7627       0.7892   
              Target Response           0.601 ± 0.1  0.555 ± 0.1  0.580 ± 0.1   
              Blood-Brain Barrier       0.780 ± 0.2  0.893 ± 0.1  0.740 ± 0.2   
              CNS MPO                   0.946 ± 0.0  0.916 ± 0.1  0.900 ± 0.1   
              Synthetic Accessibility   0.890 ± 0.0  0.934 ± 0.1  0.826 ± 0.1   
              Validity                        0.857        0.970        0.926   
              Uniqueness                      0.837        0.347        0.794   
              Novelty                         0.998        0.999        0.793   
              KL divergence                   0.472        0.482        0.868   
              Frechet ChemNet Distance        0.003        0.001        0.108   
parkinson     Score                          0.7956       0.6647       0.7967   
              Target Response           0.572 ± 0.1  0.545 ± 0.1  0.583 ± 0.1   
              Blood-Brain Barrier       0.827 ± 0.1  0.916 ± 0.1  0.706 ± 0.3   
              CNS MPO                   0.925 ± 0.1  0.890 ± 0.0  0.902 ± 0.1   
              Synthetic Accessibility   0.891 ± 0.0  0.958 ± 0.0  0.834 ± 0.1   
              Validity                        0.839        0.984        0.916   
              Uniqueness                      0.702        0.092        0.795   
              Novelty                         0.999        1.000        0.819   
              KL divergence                   0.457        0.288        0.836   
              Frechet ChemNet Distance        0.004        0.000        0.072   

Method                                      POLYGON  
Benchmark     Metric                                 
alzheimer     Score                          0.6279  
              Target Response           0.606 ± 0.0  
              Blood-Brain Barrier       0.332 ± 0.1  
              CNS MPO                   0.529 ± 0.1  
              Synthetic Accessibility   0.753 ± 0.0  
              Validity                        0.999  
              Uniqueness                      0.432  
              Novelty                         1.000  
              KL divergence                   0.027  
              Frechet ChemNet Distance        0.000  
schizophrenia Score                          0.6866  
              Target Response           0.596 ± 0.0  
              Blood-Brain Barrier       0.866 ± 0.0  
              CNS MPO                   0.664 ± 0.0  
              Synthetic Accessibility   0.607 ± 0.0  
              Validity                        0.995  
              Uniqueness                      0.378  
              Novelty                         1.000  
              KL divergence                   0.001  
              Frechet ChemNet Distance        0.000  
parkinson     Score                          0.6928  
              Target Response           0.625 ± 0.0  
              Blood-Brain Barrier       0.902 ± 0.1  
              CNS MPO                   0.720 ± 0.0  
   

In [26]:
print(
    viz_df.to_latex()
)

\begin{tabular}{llllll}
\toprule
 & Method & CoMPO-GPT & DeepLig & MTMol-GPT & POLYGON \\
Benchmark & Metric &  &  &  &  \\
\midrule
\multirow[t]{10}{*}{alzheimer} & Score & 0.7423 & 0.6463 & 0.7293 & 0.6279 \\
 & Target Response & 0.480 ± 0.1 & 0.363 ± 0.1 & 0.482 ± 0.1 & 0.606 ± 0.0 \\
 & Blood-Brain Barrier & 0.809 ± 0.1 & 0.880 ± 0.1 & 0.624 ± 0.2 & 0.332 ± 0.1 \\
 & CNS MPO & 0.974 ± 0.0 & 0.896 ± 0.1 & 0.925 ± 0.1 & 0.529 ± 0.1 \\
 & Synthetic Accessibility & 0.908 ± 0.0 & 0.949 ± 0.0 & 0.829 ± 0.1 & 0.753 ± 0.0 \\
 & Validity & 0.782 & 0.977 & 0.884 & 0.999 \\
 & Uniqueness & 0.696 & 0.147 & 0.815 & 0.432 \\
 & Novelty & 1.000 & 1.000 & 0.869 & 1.000 \\
 & KL divergence & 0.194 & 0.229 & 0.849 & 0.027 \\
 & Frechet ChemNet Distance & 0.001 & 0.001 & 0.170 & 0.000 \\
\cline{1-6}
\multirow[t]{10}{*}{schizophrenia} & Score & 0.7997 & 0.7627 & 0.7892 & 0.6866 \\
 & Target Response & 0.601 ± 0.1 & 0.555 ± 0.1 & 0.580 ± 0.1 & 0.596 ± 0.0 \\
 & Blood-Brain Barrier & 0.780 ± 0.2 & 0.893